In [2]:
APIFY_TOKEN = "apify_api_eaZDOvEu0afgstrJoh1WbCBZu0whXM4hq4f0"
DATASET_ID = "kWa39GjkO3AfBgwFU"

import requests
import pandas as pd

url = f"https://api.apify.com/v2/datasets/{DATASET_ID}/items"
params = {
    "token": APIFY_TOKEN,
    "format": "json"
}

r = requests.get(url, params=params)
r.raise_for_status()
items = r.json()

df_raw = pd.json_normalize(items)
df_raw.head()


,id,text,textLanguage,createTime,createTimeISO,isAd,webVideoUrl,mediaUrls,commentsDatasetUrl,diggCount,...,locationMeta.city,locationMeta.cityCode,locationMeta.countryCode,locationMeta.locationName,locationMeta.locationId,isMuted,authorMeta.roomId,authorMeta.commerceUserInfo.commerceUser,authorMeta.ttSeller,slideshowImageLinks
0,7415301275404733738,#salvadoreña🇸🇻 #foryou #fy #fyp #foryoupage #f...,un,1726509377,2024-09-16T17:56:17.000Z,False,https://www.tiktok.com/@661._miaa/video/741530...,[],None,183900,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7376405980554349867,#fypツ,un,1717453358,2024-06-03T22:22:38.000Z,False,https://www.tiktok.com/@chasityromann/video/73...,[],None,236600,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7399290409798651166,surpise 😝 #mexican #latina #fyp #foryou #foryo...,fr,1722781569,2024-08-04T14:26:09.000Z,False,https://www.tiktok.com/@lindceybeautyy/video/7...,[],None,404000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7374168170237660459,@🌅 #fyp #foryou #viral,un,1716932328,2024-05-28T21:38:48.000Z,False,https://www.tiktok.com/@iannelsonnnnn/video/73...,[],None,359300,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7496360083295718674,i have a feeling #fyp,en,1745382350,2025-04-23T04:25:50.000Z,False,https://www.tiktok.com/@nightmoonxyz/video/749...,[],None,2391,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
cols_map = {
    "authorMeta.name": "username",
    "diggCount": "likes",
    "commentCount": "comments",
    "shareCount": "shares",
    "playCount": "views",
    "videoMeta.duration": "duration_seconds",
    "createTimeISO": "created_at",
    "webVideoUrl": "url",
}

df = df_raw[list(cols_map.keys())].rename(columns=cols_map)
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
print(df.head(), df.shape)


         username   likes  comments  shares    views  duration_seconds  \
0       661._miaa  183900       329    3361   826700                 9   
1   chasityromann  236600       379    5518  2600000                15   
2  lindceybeautyy  404000       936    9560  3200000                12   
3   iannelsonnnnn  359300       936   11700  3500000                15   
4    nightmoonxyz    2391        99     118    48600                21   

                 created_at                                                url  
0 2024-09-16 17:56:17+00:00  https://www.tiktok.com/@661._miaa/video/741530...  
1 2024-06-03 22:22:38+00:00  https://www.tiktok.com/@chasityromann/video/73...  
2 2024-08-04 14:26:09+00:00  https://www.tiktok.com/@lindceybeautyy/video/7...  
3 2024-05-28 21:38:48+00:00  https://www.tiktok.com/@iannelsonnnnn/video/73...  
4 2025-04-23 04:25:50+00:00  https://www.tiktok.com/@nightmoonxyz/video/749...   (1248, 8)


In [4]:

df = df.dropna(subset=["created_at", "url", "views"])

num_cols = ["views", "likes", "comments", "shares", "duration_seconds"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=num_cols)
df[num_cols] = df[num_cols].astype(int)
print("After type cleaning:", df.shape)


After type cleaning: (1248, 8)


In [5]:

print(f"Original rows: {len(df)}")

df = df[
    (df["likes"]    <= df["views"]) &
    (df["comments"] <= df["views"]) &
    (df["shares"]   <= df["views"])
]
print(f"After impossible filters: {len(df)}")

min_ratio = 0.0001  # 0.01% من المشاهدات
eng_like = df["likes"] / df["views"]
eng_com  = df["comments"] / df["views"]

df = df[(eng_like >= min_ratio) | (eng_com >= min_ratio)]
print(f"After suspiciously low engagement: {len(df)}")


Original rows: 1248
After impossible filters: 1248
After suspiciously low engagement: 1246


In [6]:

before = len(df)
df = df.drop_duplicates(subset=["url"], keep="first")
print(f"Duplicates removed: {before - len(df)}")
print("Final rows:", len(df))

df["engagement_rate"] = (df["likes"] + df["comments"] + df["shares"]) / df["views"]
df["date"] = df["created_at"].dt.date

df.to_csv("tiktok_cleaned.csv", index=False, encoding="utf-8-sig")
print("✅ Cleaned data saved to tiktok_cleaned.csv")


Duplicates removed: 36
Final rows: 1210
✅ Cleaned data saved to tiktok_cleaned.csv


In [9]:
import requests
import pandas as pd
import time

SCRAPECREATORS_KEY = "LFx5lkaEn5MvOGtWYQMqRoL6RmG2"
BASE_URL = "https://api.scrapecreators.com/v1/tiktok/search/hashtag"

hashtags = [
    "fyp", "tiktok", "viral", "trending",
    "learnontiktok", "comedy", "lifestyle", "fitness"
]

PER_HASHTAG_TARGET = 400
PAGE_COUNT = 50
SLEEP_BETWEEN_CALLS = 1.0


def fetch_hashtag_page(hashtag: str, count: int = 50, cursor: int | str = 0) -> dict:
    params = {
        "hashtag": hashtag,
        "count": count,
        "cursor": cursor,
        "trim": True,
    }
    headers = {
        "x-api-key": SCRAPECREATORS_KEY,
    }
    r = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    return r.json()


all_rows = []

for tag in hashtags:
    print(f"\n===== HASHTAG: #{tag} =====")
    cursor = 0
    has_more = True
    collected = 0
    hashtag_dfs = []

    while has_more and collected < PER_HASHTAG_TARGET:
        try:
            data = fetch_hashtag_page(tag, count=PAGE_COUNT, cursor=cursor)
            has_more = bool(data.get("has_more", False))
            cursor = data.get("cursor", 0)
            videos = data.get("aweme_list", [])
            page_count = len(videos)
            print(f"cursor={cursor} | has_more={has_more} | page_count={page_count}")

            if page_count == 0:
                break

            df_page = pd.json_normalize(videos)
            df_page["source"] = "scrapecreators"
            df_page["hashtag"] = tag
            hashtag_dfs.append(df_page)
            collected += page_count
            time.sleep(SLEEP_BETWEEN_CALLS)

        except Exception as e:
            print(f"Error while fetching #{tag} page: {e}")
            break

    if hashtag_dfs:
        df_tag = pd.concat(hashtag_dfs, ignore_index=True)
        all_rows.append(df_tag)
        print(f"Collected {len(df_tag)} rows for #{tag}")
    else:
        print(f"No data collected for #{tag}")

if not all_rows:
    print("\nNo data collected for any hashtag.")
else:
    df_b_raw = pd.concat(all_rows, ignore_index=True)
    df_b_raw.to_csv("tiktok_source2_raw_paginated.csv", index=False, encoding="utf-8-sig")
    print(f"\nSaved tiktok_source2_raw_paginated.csv with {len(df_b_raw)} rows.")



===== HASHTAG: #fyp =====
cursor=20 | has_more=True | page_count=19
cursor=40 | has_more=True | page_count=18
cursor=60 | has_more=True | page_count=17
cursor=80 | has_more=True | page_count=19
cursor=100 | has_more=True | page_count=16
cursor=120 | has_more=True | page_count=17
cursor=140 | has_more=True | page_count=17
cursor=160 | has_more=True | page_count=20
cursor=180 | has_more=True | page_count=17
cursor=200 | has_more=True | page_count=18
cursor=220 | has_more=True | page_count=19
cursor=240 | has_more=True | page_count=16
cursor=260 | has_more=True | page_count=20
cursor=280 | has_more=True | page_count=19
cursor=300 | has_more=True | page_count=19
cursor=320 | has_more=True | page_count=19
cursor=340 | has_more=True | page_count=14
cursor=360 | has_more=True | page_count=19
cursor=380 | has_more=True | page_count=19
cursor=400 | has_more=True | page_count=18
cursor=420 | has_more=True | page_count=16
cursor=440 | has_more=True | page_count=15
cursor=460 | has_more=True | pa

In [16]:
import pandas as pd

# 1) Read raw ScrapeCreators file
df2 = pd.read_csv("tiktok_source2_raw_paginated.csv", low_memory=False)

# 2) Select and rename columns to match source 1
cols_map = {
    "aweme_id": "videoId",
    "desc": "text",
    "create_time": "createTime",
    "statistics.digg_count": "likes",
    "statistics.comment_count": "comments",
    "statistics.share_count": "shares",
    "statistics.play_count": "views",
    "author.unique_id": "authorUsername",
    "author.nickname": "authorName",
    "hashtag": "hashtag",
}

existing_raw = [c for c in cols_map.keys() if c in df2.columns]
df2_tmp = df2[existing_raw].rename(columns=cols_map)

# 3) Type conversions
if "createTime" in df2_tmp.columns:
    df2_tmp["createTime"] = pd.to_datetime(
        df2_tmp["createTime"], unit="s", errors="coerce"
    )

for c in ["likes", "comments", "shares", "views"]:
    if c in df2_tmp.columns:
        df2_tmp[c] = pd.to_numeric(df2_tmp[c], errors="coerce").fillna(0).astype("int64")

# 4) Build videoUrl
if "videoId" in df2_tmp.columns:
    df2_tmp["videoUrl"] = "https://www.tiktok.com/@/video/" + df2_tmp["videoId"].astype(str)

# 5) Basic cleaning: duplicates, impossible values, missing critical fields
df2_clean = df2_tmp.copy()

if "videoId" in df2_clean.columns:
    df2_clean = df2_clean.drop_duplicates(subset=["videoId"], keep="first")

for c in ["views", "likes", "comments", "shares"]:
    if c in df2_clean.columns:
        df2_clean = df2_clean[df2_clean[c] >= 0]

if all(c in df2_clean.columns for c in ["views", "likes"]):
    df2_clean = df2_clean[df2_clean["likes"] <= df2_clean["views"]]

critical_cols = ["videoId", "createTime", "hashtag"]
existing_critical = [c for c in critical_cols if c in df2_clean.columns]
if existing_critical:
    df2_clean = df2_clean.dropna(subset=existing_critical)

# 6) Column order
final_cols_order = [
    "videoUrl",
    "videoId",
    "text",
    "hashtag",
    "authorUsername",
    "authorName",
    "createTime",
    "views",
    "likes",
    "comments",
    "shares",
]

final_cols = [c for c in final_cols_order if c in df2_clean.columns]
df2_clean = df2_clean[final_cols].copy()

# 7) Save final cleaned file
df2_clean.to_csv("tiktok_source2_clean_aligned_full.csv", index=False, encoding="utf-8-sig")
